# SOA design exploration with JAX

**Design question:** which length and pump current give useful pulse gain without excessive pulse distortion or data-pattern memory?

A diode-driven semiconductor optical amplifier is a small nonlinear circuit with an internal gain reservoir. Here it becomes a workload for **compiled, batched, differentiable circuit simulation**. We build one Circulax circuit, use `jax.vmap` to solve a design grid, measure execution time with device synchronization, and differentiate a pulse-energy objective through the circuit transient.

The plotted data come from Circulax's DC and transient solvers. Python loops are used for plotting and the explicitly labeled serial timing baseline. Batched simulations are expressed as JAX transformations, without launching a simulator process per design. `vmap` exposes batch parallelism to XLA; actual execution depends on the CPU/GPU backend and is not a promise of one hardware thread per device.

The device parameters are illustrative. The main design study uses zero internal optical loss. Facet reflections, heating, and spectral gain dispersion are omitted. ASE is an output-noise approximation and does not deplete the reservoir.


The frequency-domain experiment represents a **VNA driving an MZM followed by a fixed-current SOA and a photodetector**. It compares absolute electrical S21 and an equal-receiver-power reference, then uses source-level sweeps to explain the carrier-driven response. The SOA pump is not RF-modulated in that measurement.


In [ ]:
%matplotlib inline
import time
from functools import partial

import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import brentq

from circulax import compile_circuit
from circulax.components.base_component import PhysicsReturn, Signals, source
from circulax.components.electronic import Resistor, VoltageSource
from circulax.components.soa import SemiconductorOpticalAmplifier, ase_field, soa_scales, unsaturated_gain

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.2, "figure.constrained_layout.use": True})
length_um = 600.0
reference_current_a = 0.080  # At the 600 µm reference length.
lifetime_s = 280e-12
alpha_h = 3.5
noise_figure_db = 6.5
wavelength_nm = 1540.0
bias_v = 1.8 * 0.02585 * np.log1p(reference_current_a / 1e-12) + 5 * reference_current_a
_, _, saturation_energy = map(float, soa_scales(wavelength_nm=wavelength_nm))
internal_saturation_power = saturation_energy / lifetime_s
print("JAX", jax.__version__, "| backend:", jax.default_backend(), "| devices:", jax.devices())
print(f"Reference: {length_um:g} µm, {reference_current_a * 1e3:g} mA, tau={lifetime_s * 1e12:g} ps")
print(f"Saturation energy {saturation_energy * 1e12:.3f} pJ")

## One circuit, many parameter sets

The electrical path is `anode → series resistance → diode || junction capacitance → cathode`. Conduction current pumps the gain reservoir; displacement current does not. The optical path maps input envelope to amplified output envelope. Diode saturation current and capacitance scale with contact area, and vertical series resistance scales inversely.

With active volume $V_a=wdL$, modal differential gain $a_m=\Gamma a$, and injection efficiency $\eta$,

$$h_0=a_mL\left(\frac{\eta I_D\tau_c}{qV_a}-N_{tr}\right),\qquad
\tau_c\dot h=h_0-h-\frac{P_{in}\tau_c}{E_{sat}}(e^h-1),\qquad
E_{sat}=\frac{h_P\nu wd}{a_m}.$$

The complex field is $A_{out}=A_{in}\exp[(1-j\alpha_H)h/2]$. Circulax represents the reservoir using a storage term in $F+dQ/dt=0$. The diode I–V relation drives the optical dynamics behaviorally; optical depletion does not feed back into electrical voltage.

A design vector contains **length (µm), pump-density multiplier, pulse energy (pJ), holding power (mW), lifetime (ps), Henry factor**. The multiplier is relative to 80 mA at 600 µm. Setting it to one holds contact current density fixed while length varies. Pump current scales with length; this prevents geometry changes from silently changing injection density. The transient uses a moving optical-envelope frame and omits transit delay.


In [ ]:
# A reproducible pattern includes long runs and random bits.
bits = np.r_[np.zeros(8, dtype=int), np.ones(8, dtype=int), np.random.default_rng(47).integers(0, 2, 80)]
samples_per_bit = 32
levels = np.where(bits, 1.0, 10 ** (-12 / 10))
pattern = gaussian_filter1d(np.repeat(levels, samples_per_bit), 2.5, mode="nearest")
pattern /= pattern.mean()
pattern_x = jnp.arange(pattern.size) / samples_per_bit
pattern_y = jnp.asarray(pattern)


@source(ports=("p1", "p2"), states=("i_src",), holomorphic=True)
def Envelope(
    signals: Signals,
    t: float,
    power: float = 2e-6,
    pulse_energy: float = 0.0,
    pulse_center: float = 400e-12,
    pulse_fwhm: float = 8e-12,
    bit_period: float = 125e-12,
    data: float = 0.0,
) -> PhysicsReturn:
    sigma = pulse_fwhm / jnp.sqrt(8 * jnp.log(2.0))
    pulse = pulse_energy / (sigma * jnp.sqrt(2 * jnp.pi)) * jnp.exp(-0.5 * ((t - pulse_center) / sigma) ** 2)
    modulation = (1 - data) + data * jnp.interp(t / bit_period, pattern_x, pattern_y)
    # A negligible power floor keeps sqrt derivatives finite in dark tails.
    field = jnp.sqrt(power * modulation + pulse + 1e-30)
    return {"p1": signals.i_src, "p2": -signals.i_src, "i_src": signals.p1 - signals.p2 - field}, {}


netlist = {
    "instances": {
        "SOA": {"component": "soa"},
        "BIAS": {"component": "bias"},
        "LIGHT": {"component": "light"},
        "LOAD": {"component": "load", "settings": {"R": 1.0}},
        "GND": {"component": "ground"},
    },
    "connections": {
        "GND,p1": ("BIAS,p2", "LIGHT,p2", "LOAD,p2", "SOA,cathode"),
        "BIAS,p1": "SOA,anode",
        "LIGHT,p1": "SOA,optical_in",
        "SOA,optical_out": "LOAD,p1",
    },
    "ports": {"input": "SOA,optical_in", "output": "SOA,optical_out"},
}
circuit = compile_circuit(
    netlist,
    {"soa": SemiconductorOpticalAmplifier, "bias": VoltageSource, "light": Envelope, "load": Resistor, "ground": lambda: 0},
    is_complex=True,
    backend="dense",
    rtol=1e-9,
    atol=1e-11,
)
base = {
    "BIAS.V": bias_v,
    "SOA.length_um": length_um,
    "SOA.lifetime_s": lifetime_s,
    "SOA.alpha_h": alpha_h,
    "SOA.saturation": 1.0,
    "SOA.wavelength_nm": wavelength_nm,
    "LIGHT.power": 2e-6,
    "LIGHT.pulse_energy": 0.0,
    "LIGHT.pulse_center": 400e-12,
    "LIGHT.pulse_fwhm": 12e-12,
    "LIGHT.bit_period": 125e-12,
    "LIGHT.data": 0.0,
}

In [ ]:
# The circuit topology is built once. Only numeric parameters change.
def design_params(design):
    length, density, energy_pj, hold_mw, lifetime_ps, henry = design
    contact_scale = length / 600.0
    current = reference_current_a * contact_scale * density
    voltage = 1.8 * 0.02585 * jnp.log1p(current / (1e-12 * contact_scale)) + 5 / contact_scale * current
    return {
        **base,
        "SOA.length_um": length,
        "BIAS.V": voltage,
        "SOA.lifetime_s": lifetime_ps * 1e-12,
        "SOA.alpha_h": henry,
        "LIGHT.power": hold_mw * 1e-3,
        "LIGHT.pulse_energy": energy_pj * 1e-12,
        "LIGHT.pulse_center": 60e-12,
        "LIGHT.pulse_fwhm": 8e-12,
    }


@partial(jax.jit, static_argnames=("duration", "step"))
def solve_design(design, *, duration=240e-12, step=0.25e-12):
    params = design_params(design)
    times = jnp.linspace(0.0, duration, int(round(duration / step)) + 1)
    initial = circuit.dc(params={**params, "LIGHT.pulse_energy": 0.0})
    sol = circuit.transient(
        t0=0.0, t1=duration, dt0=step, y0=initial, saveat=times, params=params, max_steps=int(round(duration / step)) + 16
    )
    return {
        "time": sol.ts,
        "input": circuit.port(sol.ys, "input"),
        "output": circuit.port(sol.ys, "output"),
        "h": jnp.real(circuit.port(sol.ys, "SOA,h")),
    }


def pulse_metrics(result):
    # Pulse experiments use zero holding power, so this is signal energy.
    pin, pout = jnp.abs(result["input"]) ** 2, jnp.abs(result["output"]) ** 2
    t = result["time"]
    ein, eout = jnp.trapezoid(pin, t), jnp.trapezoid(pout, t)
    centroid_in = jnp.trapezoid(t * pin, t) / ein
    centroid_out = jnp.trapezoid(t * pout, t) / eout
    return jnp.array([10 * jnp.log10(eout / ein), (centroid_out - centroid_in) * 1e12, eout * 1e12])


def evaluate(design):
    return pulse_metrics(solve_design(design))


reference_design = jnp.array([length_um, 1.0, 0.25, 0.0, lifetime_s * 1e12, alpha_h])
batched_solve = jax.jit(jax.vmap(solve_design))
batched_metrics = jax.jit(jax.vmap(evaluate))
print("Design columns: length_um, pump_density, energy_pJ, holding_mW, lifetime_ps, alpha_H")

## A design grid in one batched call

The three axes below vary length, pump density, and input pulse energy. Flattening the grid produces independent circuit simulations; `vmap` adds the batch axis while each circuit retains its own dynamic gain state. The maps compare useful energy gain with pulse-centroid advance. A large energy gain can come with strong reshaping, so gain alone is an incomplete design objective.


In [ ]:
lengths = jnp.linspace(320.0, 880.0, 8)
densities = jnp.linspace(0.82, 1.18, 8)
energies_pj = jnp.array([0.015, 0.15, 0.75])
ll, dd, ee = jnp.meshgrid(lengths, densities, energies_pj, indexing="ij")
designs = jnp.stack(
    [ll.ravel(), dd.ravel(), ee.ravel(), jnp.zeros(ll.size), jnp.full(ll.size, lifetime_s * 1e12), jnp.full(ll.size, alpha_h)],
    axis=1,
)
start = time.perf_counter()
metrics = batched_metrics(designs)
metrics.block_until_ready()
first_grid_seconds = time.perf_counter() - start
maps = np.asarray(metrics).reshape(len(lengths), len(densities), len(energies_pj), 3)
assert np.isfinite(maps).all()
print(f"{len(designs)} independent nonlinear transients; first call including compilation: {first_grid_seconds:.3f} s")

fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey=True)
for col, energy in enumerate(energies_pj):
    for row, (metric_index, label) in enumerate([(0, "Energy gain (dB)"), (1, "Centroid shift (ps)")]):
        image = axes[row, col].pcolormesh(
            np.asarray(densities),
            np.asarray(lengths),
            maps[:, :, col, metric_index],
            shading="nearest",
            cmap="viridis" if row == 0 else "coolwarm",
        )
        fig.colorbar(image, ax=axes[row, col], label=label)
        axes[row, col].set_title(f"Input {float(energy):g} pJ")
        axes[row, col].set_xlabel("Pump-density multiplier")
    axes[0, col].set_ylabel("Length (µm)")
    axes[1, col].set_ylabel("Length (µm)")
plt.show()

# Check batch independence against individual circuit solves at distant grid points.
indices = np.array([0, len(designs) // 2, len(designs) - 1])
for idx in indices:
    np.testing.assert_allclose(metrics[idx], evaluate(designs[idx]), rtol=2e-7, atol=1e-8)
# Step refinement on the most strongly driven design.
refined = pulse_metrics(solve_design(designs[-1], step=0.125e-12))
np.testing.assert_allclose(metrics[-1, 2], refined[2], rtol=0.003)
print("Batch/scalar agreement and pulse-energy step refinement passed.")

## Measure compilation, dispatch, and batching separately

The benchmark uses identical designs, tolerances, time grids, and returned metrics. The serial baseline is a Python loop over an already compiled scalar solver; the batched path is `jit(vmap(...))`. Both paths synchronize before timing ends. Wrapper first-call timings include compilation and may reuse lower-level kernels from earlier cells; warmed timings reuse compiled executables. We report the median of three runs and also time the complete design grid after compilation.

A batching benefit can include reduced Python/device-dispatch overhead as well as vectorized numerical work. It is hardware- and workload-dependent; a small CPU workload may show little benefit. These are local measurements, not a comparison with another simulator. A new array shape can trigger another compilation.


In [ ]:
benchmark_designs = designs[jnp.linspace(0, len(designs) - 1, 24).astype(int)]
scalar_metrics = jax.jit(evaluate)
# A separate wrapper makes its first compilation visible in this benchmark.
benchmark_batch = jax.jit(jax.vmap(evaluate))

start = time.perf_counter()
scalar_metrics(benchmark_designs[0]).block_until_ready()
scalar_first = time.perf_counter() - start
start = time.perf_counter()
benchmark_batch(benchmark_designs).block_until_ready()
batch_first = time.perf_counter() - start


def measure(call, repeats=3):
    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        jax.block_until_ready(call())
        timings.append(time.perf_counter() - start)
    return float(np.median(timings))


serial_result = jnp.stack([scalar_metrics(d) for d in benchmark_designs])
batch_result = benchmark_batch(benchmark_designs)
np.testing.assert_allclose(serial_result, batch_result, rtol=2e-7, atol=1e-8)
serial_seconds = measure(lambda: jnp.stack([scalar_metrics(d) for d in benchmark_designs]))
batch_seconds = measure(lambda: benchmark_batch(benchmark_designs))
grid_seconds = measure(lambda: batched_metrics(designs))
print(f"Scalar first call: {scalar_first:.3f} s; {len(benchmark_designs)}-case batch first call: {batch_first:.3f} s")
print(f"Warm serial loop: {serial_seconds:.4f} s; warm batch: {batch_seconds:.4f} s")
print(f"Measured serial/batch ratio: {serial_seconds / batch_seconds:.2f}×")
print(f"Full warm grid: {len(designs)} cases in {grid_seconds:.4f} s ({len(designs) / grid_seconds:.1f} cases/s)")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(["Warm serial loop", "Warm JAX batch"], [serial_seconds, batch_seconds], color=["0.55", "tab:blue"])
ax.set(xlabel=f"Wall time for {len(benchmark_designs)} circuit transients (s)", title=str(jax.devices()[0]))
plt.show()

## Differentiate the design objective through the transient

Automatic differentiation gives local sensitivities of output pulse energy to geometry, pumping, input energy, and carrier lifetime. We use dimensionless design multipliers so the gradient components can be compared. Henry factor is fixed here because it changes phase rather than energy in this model. A central finite-difference check guards against a misleading gradient.

The objective below is $\log E_{out}$. The reported derivative is with respect to the logarithm of each design variable, i.e. $\partial\log E_{out}/\partial\log x$. These local derivatives are useful for optimization; they do not replace checking the broader nonlinear response maps.


In [ ]:
def objective(log_multipliers):
    # Keep holding power at zero. Scale L, pump density, input pulse energy, tau.
    scales = jnp.exp(jnp.array([log_multipliers[0], log_multipliers[1], log_multipliers[2], 0.0, log_multipliers[3], 0.0]))
    varied = reference_design * scales
    return jnp.log(evaluate(varied)[2])


value_and_gradient = jax.jit(jax.value_and_grad(objective))
value, gradient = value_and_gradient(jnp.zeros(4))
jax.block_until_ready(gradient)
epsilon = 2e-3
basis = jnp.eye(4) * epsilon
fd = jax.vmap(lambda delta: (objective(delta) - objective(-delta)) / (2 * epsilon))(basis)
np.testing.assert_allclose(gradient, fd, rtol=0.015, atol=0.003)
print("AD sensitivities:", np.asarray(gradient))
print("Finite differences:", np.asarray(fd))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(["Length", "Pump density", "Input energy", "Carrier lifetime"], gradient, color="tab:green")
ax.axhline(0, color="black", linewidth=0.8)
ax.set(ylabel="d log(output energy) / d log(parameter)", title="Gradient through Circulax's nonlinear transient")
plt.show()

## Gradient descent chooses a device length

Instead of maximizing gain until the length bound is reached, choose a finite **target output pulse energy**. Minimize the squared relative energy error plus a small footprint penalty:

$$J(L)=\left[E_{out}(L)/E_{target}-1\right]^2+0.02(L/600\,\mu m)^2.$$

The pump **current density** stays fixed, so a longer device also draws more total current. The length penalty slightly favors shorter devices over exact target matching. This is a soft design objective, not a proof of the shortest device satisfying a hard constraint. The allowable length range is 250–950 µm.

Two initial lengths run through projected gradient descent in parallel. `value_and_grad` differentiates through the full circuit transient; `lax.scan` compiles the optimizer loop, and `vmap` batches the two starts. A dense length sweep supplies an independent check of the final objective value.

In [ ]:
target_energy_pj = 3.0


def length_loss(normalized_length):
    design = reference_design.at[0].set(600 * normalized_length)
    output_energy = evaluate(design)[2]
    return (output_energy / target_energy_pj - 1) ** 2 + 0.02 * normalized_length**2


length_value_grad = jax.value_and_grad(length_loss)


def descend(initial_length):
    def step(x, _):
        loss, gradient = length_value_grad(x)
        return jnp.clip(x - 0.08 * gradient, 250 / 600, 950 / 600), jnp.array([x * 600, loss, gradient])

    final, history = jax.lax.scan(step, initial_length / 600, xs=None, length=36)
    return final * 600, history


optimize_lengths = jax.jit(jax.vmap(descend))
start = time.perf_counter()
optimized, history = optimize_lengths(jnp.array([330.0, 900.0]))
jax.block_until_ready(optimized)
print(f"Compiled optimization including first execution: {time.perf_counter() - start:.3f} s")
start = time.perf_counter()
jax.block_until_ready(optimize_lengths(jnp.array([330.0, 900.0])))
print(f"Warm 36-step optimization, two starts: {time.perf_counter() - start:.3f} s")
check_lengths = jnp.linspace(250, 950, 81)
check_losses = jax.jit(jax.vmap(length_loss))(check_lengths / 600)
final_losses = jax.vmap(length_loss)(optimized / 600)
np.testing.assert_allclose(optimized[0], optimized[1], atol=1.0)
assert np.all(np.asarray(final_losses) < np.min(check_losses) + 0.001)
assert np.all(np.asarray(final_losses) < np.asarray(history[:, 0, 1]))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for k, initial in enumerate([330, 900]):
    axes[0].plot(np.asarray(history[k, :, 0]), label=f"Start {initial} µm")
    axes[1].semilogy(np.asarray(history[k, :, 1]))
axes[0].set(xlabel="Gradient-descent iteration", ylabel="Length (µm)")
axes[0].legend()
axes[1].set(xlabel="Gradient-descent iteration", ylabel="Objective")
axes[2].plot(check_lengths, check_losses, label="Independent length sweep")
axes[2].scatter(optimized, final_losses, color="tab:red", label="Gradient descent")
axes[2].set(xlabel="Length (µm)", ylabel="Objective")
axes[2].legend()
plt.show()
for optimum in np.asarray(optimized):
    d = reference_design.at[0].set(optimum)
    print(f"Length {optimum:.2f} µm → output {float(evaluate(d)[2]):.3f} pJ; target {target_energy_pj:g} pJ")

## Inspect selected trajectories: amplitude, recovery, and phase

The grid summarizes many designs, but selected trajectories explain the mechanism. The first panel varies pulse energy at fixed geometry and pump density. The second varies holding power to expose the gain/recovery tradeoff. The third compares a zero-Henry-factor control with carrier-induced chirp; the fourth shows the associated spectra. Each family is evaluated with `vmap`.

We use the physical field convention Re{A(t) exp(−j2πν₀t)}. Thus physical frequency offset is −dφ/dt / (2π), and the FFT envelope frequency axis is negated. Chirp is checked against α_H dh/dt / (4π) only where the pulse has appreciable power.


In [ ]:
pulse_family = jnp.tile(reference_design, (3, 1)).at[:, 2].set(jnp.array([0.015, 0.15, 0.75]))
pulses = batched_solve(pulse_family)
recovery_family = jnp.tile(reference_design, (2, 1)).at[:, 3].set(jnp.array([0.002, 0.7]))
recovery_run = jax.jit(jax.vmap(lambda d: solve_design(d, duration=1.4e-9, step=1e-12)))
recovery = recovery_run(recovery_family)
phase_family = jnp.tile(reference_design, (3, 1)).at[:, 5].set(jnp.array([0.0, 2.0, 4.5]))
phase_runs = batched_solve(phase_family)
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for i, design in enumerate(np.asarray(pulse_family)):
    t = np.asarray(pulses["time"][i])
    p = np.abs(pulses["output"][i]) ** 2
    axes[0, 0].plot((t - 60e-12) * 1e12, p / p.max(), label=f"{design[2]:g} pJ")
axes[0, 0].set(xlim=(-18, 18), xlabel="Time from pulse center (ps)", ylabel="Peak-normalized output")
axes[0, 0].legend()
for i, hold in enumerate(np.asarray(recovery_family[:, 3])):
    axes[0, 1].plot(
        (np.asarray(recovery["time"][i]) - 60e-12) * 1e12,
        np.asarray(recovery["h"][i]) * 10 / np.log(10),
        label=f"Holding {hold:g} mW",
    )
axes[0, 1].set(xlim=(-30, 1200), xlabel="Time from pulse center (ps)", ylabel="Gain (dB)")
axes[0, 1].legend()
for i, henry in enumerate(np.asarray(phase_family[:, 5])):
    t = np.asarray(phase_runs["time"][i])
    field = np.asarray(phase_runs["output"][i])
    p = np.abs(field) ** 2
    mask = p > p.max() * 0.01
    chirp = -np.gradient(np.unwrap(np.angle(field)), t) / (2 * np.pi)
    check = henry * np.gradient(np.asarray(phase_runs["h"][i]), t) / (4 * np.pi)
    np.testing.assert_allclose(chirp[mask], check[mask], rtol=1e-7, atol=1.0)
    axes[1, 0].plot((t[mask] - 60e-12) * 1e12, chirp[mask] / 1e9, label=f"α_H = {henry:g}")
    frequencies = -np.fft.fftshift(np.fft.fftfreq(len(t), t[1] - t[0])) / 1e9
    spectrum = np.abs(np.fft.fftshift(np.fft.fft(field))) ** 2
    order = np.argsort(frequencies)
    axes[1, 1].plot(frequencies[order], (spectrum / spectrum.max())[order])
axes[1, 0].set(xlabel="Time from pulse center (ps)", ylabel="Physical frequency offset (GHz)")
axes[1, 0].legend()
axes[1, 1].set(xlim=(-200, 140), xlabel="Physical frequency offset (GHz)", ylabel="Normalized spectral density")
plt.show()

## Batched data-pattern stress test

A second batch tests data signals instead of isolated pulses. Rates and mean optical powers form a Cartesian grid, with 64 samples per bit at every rate. Every case receives the same reproducible bit pattern. We discard sixteen settling bits, then measure the logical-one power spread at bit centers. Eye diagrams show two selected cases; the map shows the full rate/power tradeoff.


In [ ]:
def data_case(rate_and_power):
    rate, power_mw = rate_and_power
    period = 1 / rate
    count = len(bits) * 64
    step = period / 64
    params = {
        **design_params(reference_design),
        "LIGHT.data": 1.0,
        "LIGHT.pulse_energy": 0.0,
        "LIGHT.bit_period": period,
        "LIGHT.power": power_mw * 1e-3,
    }
    initial = circuit.dc(params=params)
    ts = jnp.arange(count + 1) * step
    sol = circuit.transient(t0=0.0, t1=count * step, dt0=step, y0=initial, saveat=ts, params=params, max_steps=count + 16)
    p = jnp.abs(circuit.port(sol.ys, "output")) ** 2
    centers = p[jnp.arange(16, len(bits)) * 64 + 32]
    # Mask logical zeros without introducing dynamic-size arrays under JIT.
    ones_mask = jnp.asarray(bits[16:]) == 1
    low = jnp.min(jnp.where(ones_mask, centers, jnp.inf))
    high = jnp.max(jnp.where(ones_mask, centers, -jnp.inf))
    return p[16 * 64 : -1], 10 * jnp.log10(high / low)


rates = jnp.array([3e9, 6e9, 12e9])
mean_powers_mw = jnp.array([0.01, 0.05, 0.15, 0.5])
rr, pp = jnp.meshgrid(rates, mean_powers_mw, indexing="ij")
data_designs = jnp.stack([rr.ravel(), pp.ravel()], axis=1)
data_batch = jax.jit(jax.vmap(data_case))
data_power, one_spread = data_batch(data_designs)
jax.block_until_ready(one_spread)
assert np.isfinite(one_spread).all()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.7))
for ax, index in zip(axes[:2], [2, 10]):
    p = np.array(data_power[index])
    p /= p.mean()
    for start in range(0, len(p) - 128, 64):
        ax.plot(np.arange(128) / 64, p[start : start + 128], color="tab:purple", alpha=0.14)
    ax.set(
        title=f"{float(data_designs[index, 0]) / 1e9:g} Gb/s, {float(data_designs[index, 1]):g} mW",
        xlabel="Time (unit intervals)",
        ylabel="Output / mean power",
    )
image = axes[2].imshow(np.asarray(one_spread).reshape(len(rates), len(mean_powers_mw)), aspect="auto", origin="lower", cmap="magma")
axes[2].set_xticks(range(len(mean_powers_mw)), [f"{float(p):g}" for p in mean_powers_mw])
axes[2].set_yticks(range(len(rates)), [f"{float(r) / 1e9:g}" for r in rates])
axes[2].set(xlabel="Mean input power (mW)", ylabel="Bit rate (Gb/s)")
fig.colorbar(image, ax=axes[2], label="One-level spread (dB)")
plt.show()
print(f"Solved {len(data_designs)} data-pattern cases in one JAX batch.")

## Does a long SOA need spatial segments?

Length alone does not decide this. With a single forward wave, linear material gain, uniform carrier lifetime, no passive loss and a moving time coordinate, integrating over length closes the gain equation. In that special case, a chain of cells has the same output as one integrated reservoir: stimulated-emission extraction telescopes as $\sum(P_{j+1}-P_j)=P_{out}-P_{in}$. Local carriers can still be nonuniform; one state simply cannot display their spatial profile.

With **distributed passive loss**, photon extraction from carriers is no longer the net output-minus-input power. Local recombination, nonuniform lifetime, independently varying pump sections, counterpropagating signals, and ASE depletion can also require spatial resolution. A traveling-wave model is needed when resolving propagation and bidirectional fields. See [Ó Dúill and Barry, *Improved Reduced Models for Single-Pass and Reflective Semiconductor Optical Amplifiers*](https://arxiv.org/abs/1407.5579) for the role of internal loss in subdivided SOA models.

Here each section has its own diode and gain state, and equal electrical bias maintains uniform contact current density. For section gain $h_j$ and passive attenuation exponent $\ell_j$, the field multiplier is $\exp[(h_j-\ell_j-j\alpha_Hh_j)/2]$. Carrier extraction is proportional to $h_j\,\mathrm{exprel}(h_j-\ell_j)P_{in,j}$, where $\mathrm{exprel}(x)=(e^x-1)/x$. This approximates constant gain within each section while distinguishing passive absorption from stimulated emission.

We compare 1, 2, 4, and 8 sections, holding total length, cross-section, current density, and timestep fixed. The lossless result is an invariant check; the lossy case is a mesh-convergence study. The final panel shows a longitudinal gain snapshot after depletion. **These cells have no propagation delay**: they resolve longitudinal carrier dynamics in the moving frame. At group index 3.6, a 1.4 mm device has about 17 ps transit time; add propagation delays/a traveling-wave treatment for laboratory-time arrival or rapidly varying pump signals. Segment count must be a static topology choice, so each topology is compiled separately.

In [ ]:
def segmented_fixture(sections):
    instances = {
        "BIAS": {"component": "bias"},
        "LIGHT": {"component": "light"},
        "LOAD": {"component": "load", "settings": {"R": 1.0}},
        "GND": {"component": "ground"},
    }
    for k in range(sections):
        instances[f"S{k}"] = {"component": "soa"}
    connections = {
        "GND,p1": tuple(["BIAS,p2", "LIGHT,p2", "LOAD,p2"] + [f"S{k},cathode" for k in range(sections)]),
        "BIAS,p1": tuple(f"S{k},anode" for k in range(sections)),
        "LIGHT,p1": "S0,optical_in",
        f"S{sections - 1},optical_out": "LOAD,p1",
    }
    connections.update({f"S{k},optical_out": f"S{k + 1},optical_in" for k in range(sections - 1)})
    return compile_circuit(
        {"instances": instances, "connections": connections, "ports": {"output": f"S{sections - 1},optical_out"}},
        {"soa": SemiconductorOpticalAmplifier, "bias": VoltageSource, "light": Envelope, "load": Resistor, "ground": lambda: 0},
        is_complex=True,
        backend="dense",
        rtol=1e-9,
        atol=1e-11,
    )


long_length = 1400.0
section_counts = [1, 2, 4, 8]
section_energy = {0.0: [], 18.0: []}
profiles = {}
for count in section_counts:
    segmented = segmented_fixture(count)

    @jax.jit
    def section_run(loss):
        params = {
            "length_um": long_length / count,
            "loss_db_per_cm": loss,
            "BIAS.V": bias_v,
            "lifetime_s": lifetime_s,
            "alpha_h": alpha_h,
            "LIGHT.power": 0.0,
            "LIGHT.pulse_energy": 0.3e-12,
            "LIGHT.pulse_center": 60e-12,
            "LIGHT.pulse_fwhm": 8e-12,
        }
        initial = segmented.dc(params={**params, "LIGHT.pulse_energy": 0.0})
        ts = jnp.linspace(0.0, 200e-12, 401)
        sol = segmented.transient(t0=0.0, t1=200e-12, dt0=0.5e-12, y0=initial, saveat=ts, params=params, max_steps=416)
        field = segmented.port(sol.ys, "output")
        gains = jnp.stack([jnp.real(segmented.port(sol.ys, f"S{k},h")) for k in range(count)], axis=1)
        return jnp.trapezoid(jnp.abs(field) ** 2, ts) * 1e12, ts, gains

    # Batch loss settings within each fixed topology.
    energy, ts, gains = jax.vmap(section_run)(jnp.array([0.0, 18.0]))
    for k, loss in enumerate([0.0, 18.0]):
        section_energy[loss].append(float(energy[k]))
    profiles[count] = (np.asarray(ts[1]), np.asarray(gains[1]))
np.testing.assert_allclose(section_energy[0.0], section_energy[0.0][0], rtol=0.003)
lossy_error_db = 10 * np.log10(np.asarray(section_energy[18.0]) / section_energy[18.0][-1])
assert abs(lossy_error_db[-2]) < 0.1
fig, axes = plt.subplots(1, 3, figsize=(13, 3.7))
for loss, values in section_energy.items():
    axes[0].plot(section_counts, values, "o-", label=f"Passive loss {loss:g} dB/cm")
axes[0].set(xlabel="Number of sections", ylabel="Output pulse energy (pJ)")
axes[0].legend()
axes[1].plot(section_counts, lossy_error_db, "o-")
axes[1].set(xlabel="Number of sections", ylabel="Energy difference from 8 sections (dB)")
ts, gain_profile = profiles[8]
for instant in [45e-12, 60e-12, 85e-12]:
    idx = np.searchsorted(ts, instant)
    axes[2].plot(
        (np.arange(8) + 0.5) * long_length / 8,
        gain_profile[idx] / (long_length / 8 * 1e-6),
        "o-",
        label=f"t = {instant * 1e12:g} ps",
    )
axes[2].set(xlabel="Position (µm)", ylabel="Active modal gain (1/m)")
axes[2].legend()
plt.show()
print("Lossy energy differences relative to 8 sections (dB):", lossy_error_db)
print("Transit time at n_g=3.6:", 3.6 * long_length * 1e-6 / 299792458 * 1e12, "ps")

## VNA characterization: MZM → fixed-current SOA → photodetector

This is an **electro-optic small-signal link measurement**. The RF drive modulates an MZM; it does not modulate the SOA current. A 50 Ω source and matched 50 Ω modulator termination define the input reference plane. An ideal square-law photodiode drives a 50 Ω receiver. With equal reference impedances, $S_{21}=2\delta V_{receiver}/V_{source,open}$, a dimensionless electrical wave ratio. The factor of two accounts for the source voltage division.

The MZM is biased at quadrature with a 4 V half-wave voltage. Its compact dynamics use Circulax’s `Bessel3` electrical-drive filter, followed by the static interferometer transfer. The default third-order Bessel cutoff is 35 GHz, exposed as `mzi_filter_f3db`. The matched 50 Ω termination stays at the RF input reference plane; the internal filter is a buffered behavioral block, not a passive RF filter. The SOA is supplied by an independent DC current source. Each CW operating point is linearized with `Circuit.small_signal()`, including the real/imaginary optical coordinates and the SOA carrier state. No finite optical modulation depth is used. The detector is ideal and wideband in this experiment. Both the MZM-only reference and the MZM–SOA path contain the same third-order drive filter, so de-embedding cancels its response while the absolute link S21 retains its bandwidth and phase delay. This is a generic compact MZI model, without PDK-specific PN transfer tables or junction RC parameters.

Two comparisons are shown. The absolute electrical $S_{21}$ includes average optical amplification. A second comparison attenuates the SOA output so that the detector sees the same mean optical power as the modulator-only reference. This removes average amplification and reveals the SOA's effect on modulation depth. Normalizing each curve to its own low-frequency value would obscure the suppression we want to examine.

Slow optical intensity modulation drives carrier-density oscillations even with a constant pump current. The accompanying gain oscillation opposes the input intensity modulation and suppresses detected RF output. At higher RF frequency the carriers cannot follow, so the modulation approaches the frozen-gain limit. This is already small-signal physics. A related measured modulator/SOA link and equal-receiver-power normalization are reported by [Moscoso-Mártir et al., *Calibrated Link Budget of a Silicon Photonics WDM Transceiver with SOA and Semiconductor Mode-Locked Laser*](https://pmc.ncbi.nlm.nih.gov/articles/PMC5607229/).

The comparison below does not yet include a wavelength-selective filter or fiber dispersion. Such components can convert carrier-induced phase modulation into additional detected intensity features; attributing a specific measured notch to them needs the actual link configuration.


For this confirmed MZM setup, the first mechanism to examine is the optically driven gain oscillation already represented by the carrier ODE. No thermal or carrier-capture state has been added merely to create a dip. The model predicts a smooth RF shelf and phase excursion. A sharper measured notch would require checking modulator chirp, optical filtering/dispersion, receiver bandwidth, and calibration before attributing it to additional SOA physics.


The independent filter benchmark is $H_B(s)=15/(p^3+6p^2+15p+15)$ with $p=s/\omega_B$ and $\omega_B=2\pi f_{3dB}/1.7556723686812106$. We verify the complex reference response and the exact −3.0103 dB point, then sweep the MZI cutoff while keeping the SOA operating point fixed. This demonstrates the separate origins of low-frequency carrier suppression and high-frequency modulator roll-off.


In [ ]:
from circulax.components.base_component import component
from circulax.components.electronic import CurrentSource
from circulax.components.filters import Bessel3
from circulax.components.photonic import OpticalSource

mzi_filter_f3db = 35e9


@component(ports=("ein", "eout", "control"), states=("i_out",), holomorphic=False)
def VNAModulator(signals: Signals, vpi: float = 4.0, bias_phase: float = np.pi / 2) -> PhysicsReturn:
    phase = bias_phase + jnp.pi * jnp.real(signals.control) / vpi
    field = signals.ein * jnp.cos(phase / 2)
    return {"ein": signals.ein, "control": 0.0, "eout": signals.i_out, "i_out": signals.eout - field}, {}


@component(ports=("ein", "eout"), states=("i_out",), holomorphic=True)
def VNAOpticalAttenuator(signals: Signals, transmission: float = 1.0) -> PhysicsReturn:
    return {"ein": signals.ein, "eout": signals.i_out, "i_out": signals.eout - jnp.sqrt(transmission) * signals.ein}, {}


@component(ports=("ein", "out", "ground"), holomorphic=False)
def VNADetector(signals: Signals, responsivity: float = 0.8) -> PhysicsReturn:
    current = responsivity * jnp.abs(signals.ein) ** 2
    return {"ein": signals.ein, "out": -current, "ground": current}, {}


def vna_fixture(include_soa):
    instances = {
        "GND": {"component": "ground"},
        "LASER": {"component": "laser"},
        "RF": {"component": "voltage"},
        "R_SOURCE": {"component": "resistor", "settings": {"R": 50.0}},
        "R_MOD": {"component": "resistor", "settings": {"R": 50.0}},
        "MZM": {"component": "mzm"},
        "MZM_BW": {"component": "bessel3", "settings": {"f3db": mzi_filter_f3db}},
        "ATT": {"component": "attenuator"},
        "PD": {"component": "detector"},
        "R_RX": {"component": "resistor", "settings": {"R": 50.0}},
    }
    ground_nodes = ["LASER,p2", "RF,p2", "R_MOD,p2", "PD,ground", "R_RX,p2"]
    connections = {
        "LASER,p1": "MZM,ein",
        "RF,p1": "R_SOURCE,p1",
        "R_SOURCE,p2": ("MZM_BW,inp", "R_MOD,p1"),
        "MZM_BW,out": "MZM,control",
        "ATT,eout": "PD,ein",
        "PD,out": "R_RX,p1",
    }
    if include_soa:
        instances.update({"SOA": {"component": "soa"}, "PUMP": {"component": "current"}})
        ground_nodes.extend(["SOA,cathode", "PUMP,p1"])
        connections.update({"PUMP,p2": "SOA,anode", "MZM,eout": "SOA,optical_in", "SOA,optical_out": "ATT,ein"})
    else:
        connections["MZM,eout"] = "ATT,ein"
    connections["GND,p1"] = tuple(ground_nodes)
    return compile_circuit(
        {"instances": instances, "connections": connections, "ports": {"receiver": "PD,out"}},
        {
            "laser": OpticalSource,
            "voltage": VoltageSource,
            "resistor": Resistor,
            "mzm": VNAModulator,
            "bessel3": Bessel3,
            "attenuator": VNAOpticalAttenuator,
            "detector": VNADetector,
            "soa": SemiconductorOpticalAmplifier,
            "current": CurrentSource,
            "ground": lambda: 0,
        },
        is_complex=True,
        backend="dense",
        rtol=1e-10,
        atol=1e-12,
    )


vna_soa = vna_fixture(True)
vna_reference = vna_fixture(False)
vna_freqs = jnp.sort(jnp.r_[0.0, jnp.geomspace(1e6, 80e9, 321), mzi_filter_f3db])
vna_open_voltage = 1e-3
vna_input_powers = jnp.array([2e-6, 2e-4, 1e-3])


def vna_case(power, filter_f3db=mzi_filter_f3db):
    params = {
        "LASER.power": 2 * power,
        "MZM_BW.f3db": filter_f3db,
        "PUMP.I": reference_current_a * length_um / 600,
        "SOA.length_um": length_um,
        "SOA.lifetime_s": lifetime_s,
        "SOA.alpha_h": alpha_h,
    }
    model = vna_soa.small_signal(params=params)
    result = model(freqs=vna_freqs, sources={"RF": vna_open_voltage})
    reference_model = vna_reference.small_signal(params={"LASER.power": 2 * power, "MZM_BW.f3db": filter_f3db})
    reference_result = reference_model(freqs=vna_freqs, sources={"RF": vna_open_voltage})
    gain = jnp.abs(vna_soa.port(model.y_dc, "SOA,optical_out")) ** 2 / power
    equal_power_model = vna_soa.small_signal(params={**params, "ATT.transmission": 1 / gain})
    equal_power = equal_power_model(freqs=vna_freqs, sources={"RF": vna_open_voltage})
    s21 = 2 * result.voltage("receiver") / vna_open_voltage
    s21_ref = 2 * reference_result.voltage("receiver") / vna_open_voltage
    s21_equal = 2 * equal_power.voltage("receiver") / vna_open_voltage
    carrier = result.quadratures("SOA,h")[0] / vna_open_voltage
    junction = result.voltage("SOA,v_j")
    # DC current and its RF perturbation remain independent of optical drive.
    diode_current = 1e-12 * (length_um / 600) * jnp.expm1(jnp.real(vna_soa.port(model.y_dc, "SOA,v_j")) / (1.8 * 0.02585))
    detector_dc = vna_soa.port(equal_power_model.y_dc, "receiver")
    reference_dc = vna_reference.port(reference_model.y_dc, "receiver")
    return s21, s21_ref, s21_equal, carrier, junction, diode_current, detector_dc, reference_dc, gain


vna_results = jax.jit(jax.vmap(vna_case))(vna_input_powers)
s21, s21_reference, s21_equal, gain_modulation, junction_rf, pump_current, detector_dc, reference_dc, vna_gains = vna_results
np.testing.assert_allclose(pump_current, reference_current_a * length_um / 600, rtol=1e-6)


def bessel_benchmark(freqs, cutoff):
    p = 1j * freqs / (cutoff / 1.7556723686812106)
    return 15 / (p**3 + 6 * p**2 + 15 * p + 15)


filter_benchmark = bessel_benchmark(vna_freqs, mzi_filter_f3db)
expected_reference_s21 = -50 * 0.8 * jnp.pi / 4 * vna_input_powers[:, None] * filter_benchmark
np.testing.assert_allclose(abs(bessel_benchmark(mzi_filter_f3db, mzi_filter_f3db)), 1 / np.sqrt(2), rtol=1e-12)
np.testing.assert_allclose(s21_reference, jnp.broadcast_to(expected_reference_s21, s21_reference.shape), rtol=1e-7, atol=1e-12)
np.testing.assert_allclose(junction_rf, 0, atol=1e-12)
np.testing.assert_allclose(detector_dc, reference_dc, rtol=1e-6, atol=1e-12)
# Compare the complete electrical measurement with the independent carrier formula.
r = 1 / lifetime_s + vna_input_powers[:, None] * vna_gains[:, None] / saturation_energy
analytic_link_ratio = 1 - vna_input_powers[:, None] * (vna_gains[:, None] - 1) / saturation_energy / (r + 2j * jnp.pi * vna_freqs)
np.testing.assert_allclose(s21_equal / s21_reference, analytic_link_ratio, rtol=2e-6, atol=1e-8)
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
fghz = np.asarray(vna_freqs[1:]) / 1e9
for k, power in enumerate(vna_input_powers):
    label = f"SOA input {float(power) * 1e3:g} mW"
    color = f"C{k}"
    axes[0, 0].semilogx(fghz, 20 * np.log10(np.abs(s21[k, 1:])), color=color, label=label)
    axes[0, 0].semilogx(fghz, 20 * np.log10(np.abs(s21_reference[k, 1:])), "--", color=color, alpha=0.55)
    ratio = np.asarray(s21_equal[k] / s21_reference[k])
    axes[0, 1].semilogx(fghz, 20 * np.log10(np.abs(ratio[1:])), color=color, label=label)
    axes[1, 0].semilogx(fghz, np.unwrap(np.angle(ratio[1:])) * 180 / np.pi, color=color)
    axes[1, 1].semilogx(fghz, np.abs(gain_modulation[k, 1:]), color=color)
    print(
        f"SOA input {float(power) * 1e3:g} mW: low-frequency RF suppression "
        f"{20 * np.log10(abs(ratio[0])):.2f} dB at equal detector power; pump {float(pump_current[k]) * 1e3:.2f} mA"
    )
axes[0, 0].set(
    xlabel="VNA frequency (GHz)", ylabel="Electrical |S21| (dB)", title="35 GHz Bessel MZI: solid + SOA; dashed reference"
)
axes[0, 0].legend()
axes[0, 1].set(
    xlabel="VNA frequency (GHz)", ylabel="SOA link / reference (dB)", title="Equal mean detector power: low-frequency suppression"
)
axes[0, 1].axhline(0, color="black", linewidth=0.8)
axes[0, 1].legend()
axes[1, 0].set(xlabel="VNA frequency (GHz)", ylabel="Relative RF phase (degrees)")
axes[1, 1].set(
    xlabel="VNA frequency (GHz)",
    ylabel="|delta h / RF source voltage| (1/V)",
    title="Optically driven gain oscillation at constant current",
)
plt.show()
print("Matched VNA normalization, equal detector power, constant pump current, and carrier-response checks passed.")

# Vary the MZI bandwidth while holding optical loading and pump current fixed.
cutoffs = jnp.array([10e9, 20e9, 35e9])
cutoff_results = jax.jit(jax.vmap(lambda cutoff: vna_case(vna_input_powers[1], cutoff)))(cutoffs)
cutoff_reference = cutoff_results[1]
cutoff_equal = cutoff_results[2]
deembedded = cutoff_equal / cutoff_reference
np.testing.assert_allclose(deembedded, jnp.broadcast_to(deembedded[0], deembedded.shape), rtol=2e-6, atol=1e-8)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for k, cutoff in enumerate(cutoffs):
    label = f"MZI cutoff {float(cutoff) / 1e9:g} GHz"
    # Normalize all curves to the matching MZI-only DC response, preserving
    # SOA suppression rather than normalizing it away at low frequency.
    combined = cutoff_equal[k] / cutoff_reference[k, 0]
    axes[0].semilogx(fghz, 20 * np.log10(np.abs(combined[1:])), label=label)
    axes[1].semilogx(fghz, 20 * np.log10(np.abs(deembedded[k, 1:])), label=label)
axes[0].set(
    xlabel="VNA frequency (GHz)",
    ylabel="Link / MZI-only DC reference (dB)",
    title="Carrier suppression + third-order MZI bandwidth",
)
axes[1].set(
    xlabel="VNA frequency (GHz)",
    ylabel="SOA / matched MZI reference (dB)",
    title="De-embedded SOA response is independent of MZI cutoff",
)
axes[0].legend()
axes[1].legend()
plt.show()
print("Bessel reference magnitude/phase, exact cutoff, and MZI-bandwidth de-embedding checks passed.")

## Diagnostic: separate optical loading from electrical pump modulation

A modulation frequency should be compared with a **rate**, not directly with a time. The nominal recovery corner is $1/(2\pi\tau_c)$. Under CW loading, stimulated emission shortens the local effective recovery time:

$$r=\tau_{eff}^{-1}=\tau_c^{-1}+\bar P_{in}\bar G/E_{sat},\qquad f_c=r/(2\pi).$$

There are three distinct time scales:

- **Carrier recovery:** the gain-reservoir state already present in this model.
- **Electrical carrier transport/capture:** a separate injection process; this model has no transport/capture state. The diode's series resistance and capacitance are included, but they are not a substitute for carrier transport physics.
- **Optical transit:** $\tau_{opt}=n_gL/c$. The main model uses a moving optical-envelope frame. For uniform, time-independent pumping, a pure propagation delay can map that result back to laboratory arrival time. Distributed time-dependent pumping, reflections, or counterpropagating fields require a richer traveling-wave treatment.

The response also depends on **what is modulated**. Optical input-power modulation experiences compression at low frequency: carriers follow the drive and reduce the gain as power rises. Above the recovery corner, the carriers are nearly frozen, so this negative feedback weakens. The normalized intensity response has a shelving/high-pass form, not the low-pass shape of the carrier state itself. Electrical pump modulation must move the carrier population, so its optical response rolls off above the recovery corner.

For zero passive loss, linearizing the gain ODE gives an independent benchmark:

$$H_{opt}(\omega)=\frac{\delta P_{out}}{\bar G\delta P_{in}}
=1-\frac{\bar P_{in}(\bar G-1)/E_{sat}}{r+j\omega}.$$

This is the transmitted optical signal's modulation response; cross-gain modulation of a separate probe is a different measurement. As $\omega\to0$ it equals the DC differential gain divided by $\bar G$, and as $\omega\to\infty$ it approaches one within this model. The latter is not a prediction of unlimited physical bandwidth: ultrafast gain compression, dispersion, transport, and package parasitics are outside its scope. Carrier-driven phase changes are retained by the complex-field model.

The electrical-pump curves in this diagnostic are a separate control experiment; they do not describe the fixed-current VNA measurement above. The following batch uses Circulax's public `Circuit.small_signal()` API and its physical power-modulation observable, which retains both optical quadratures. It does not treat a non-holomorphic optical field as a single complex electrical phasor. Nine CW operating points vary lifetime and holding power while keeping the unsaturated gain fixed by adjusting pump current. Frequencies include DC and sweep from 1 MHz to 100 GHz. A dotted propagation-delay overlay changes phase, not magnitude; it is explicitly not a simulated transport state.

Carrier-lifetime-dependent modulation measurements are discussed by [Lee et al., *Measurement of the carrier lifetime and linewidth enhancement factor of semiconductor optical amplifiers using their optical modulation responses*](https://yonsei.elsevierpure.com/en/publications/measurement-of-the-carrier-lifetime-and-linewidth-enhancement-fac/).


In [ ]:
frequency_circuit = compile_circuit(
    netlist,
    {"soa": SemiconductorOpticalAmplifier, "bias": VoltageSource, "light": OpticalSource, "load": Resistor, "ground": lambda: 0},
    is_complex=True,
    backend="dense",
    rtol=1e-10,
    atol=1e-12,
)
frequencies = jnp.r_[0.0, jnp.geomspace(1e6, 1e11, 241)]
lifetimes_ps = jnp.array([200.0, 350.0, 600.0])
cw_powers_w = jnp.array([2e-6, 2e-4, 1e-3])
tt, pp = jnp.meshgrid(lifetimes_ps, cw_powers_w, indexing="ij")
frequency_cases = jnp.stack([tt.ravel(), pp.ravel()], axis=1)


def frequency_params(tau_ps, power):
    # Hold unsaturated gain fixed when changing lifetime.
    current = reference_current_a * (length_um / 600) * lifetime_s / (tau_ps * 1e-12)
    scale = length_um / 600
    voltage = 1.8 * 0.02585 * jnp.log1p(current / (1e-12 * scale)) + 5 / scale * current
    return {
        "BIAS.V": voltage,
        "LIGHT.power": power,
        "SOA.length_um": length_um,
        "SOA.lifetime_s": tau_ps * 1e-12,
        "SOA.alpha_h": alpha_h,
    }


def frequency_case(case):
    tau_ps, power = case
    model = frequency_circuit.small_signal(params=frequency_params(tau_ps, power))
    # The optical source stamp is a field perturbation in sqrt(W).
    # This amplitude corresponds to a fractional input POWER modulation of 0.001.
    optical = model(freqs=frequencies, sources={"LIGHT": 0.001 * jnp.sqrt(power) / 2})
    bias = model(freqs=frequencies, sources={"BIAS": 1e-3})
    dc_field = frequency_circuit.port(model.y_dc, "output")
    gain = jnp.abs(dc_field) ** 2 / power
    optical_response = optical.power("output", field_unit="sqrt(W)") / (gain * optical.power("input", field_unit="sqrt(W)"))
    bias_power = bias.power("output", field_unit="sqrt(W)")
    r = 1 / (tau_ps * 1e-12) + power * gain / saturation_energy
    benchmark = 1 - power * (gain - 1) / saturation_energy / (r + 2j * jnp.pi * frequencies)
    return optical_response, bias_power / bias_power[0], benchmark, r / (2 * jnp.pi), gain


frequency_batch = jax.jit(jax.vmap(frequency_case))
optical_rf, pump_rf, benchmark_rf, carrier_corners, cw_gains = frequency_batch(frequency_cases)
jax.block_until_ready(optical_rf)
np.testing.assert_allclose(optical_rf, benchmark_rf, rtol=2e-6, atol=1e-8)
assert np.all(np.isfinite(np.asarray(pump_rf)))
# DC differential-gain check on the medium-lifetime, medium-power case.
case_index = 4
params = frequency_params(*frequency_cases[case_index])
small_power_step = float(frequency_cases[case_index, 1]) * 1e-4
upper = frequency_circuit.dc(params={**params, "LIGHT.power": params["LIGHT.power"] + small_power_step})
lower = frequency_circuit.dc(params={**params, "LIGHT.power": params["LIGHT.power"] - small_power_step})
differential = (abs(frequency_circuit.port(upper, "output")) ** 2 - abs(frequency_circuit.port(lower, "output")) ** 2) / (
    2 * small_power_step
)
np.testing.assert_allclose(optical_rf[case_index, 0], differential / cw_gains[case_index], rtol=1e-6)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
ghz = np.asarray(frequencies[1:]) / 1e9
for index in [3, 4, 5]:
    label = f"CW {float(frequency_cases[index, 1]) * 1e3:g} mW"
    axes[0, 0].semilogx(ghz, 20 * np.log10(np.abs(optical_rf[index, 1:])), label=label)
    axes[0, 1].semilogx(ghz, 20 * np.log10(np.abs(pump_rf[index, 1:])), label=label)
    corner = float(carrier_corners[index]) / 1e9
    axes[0, 0].axvline(corner, color=f"C{index - 3}", linestyle=":", alpha=0.5)
axes[0, 0].set(
    xlabel="Modulation frequency (GHz)",
    ylabel="Optical response / frozen-gain limit (dB)",
    title="Input-power modulation: low-frequency compression",
)
axes[0, 1].set(
    xlabel="Modulation frequency (GHz)",
    ylabel="Pump response / DC response (dB)",
    title="Bias-voltage modulation: carrier-limited roll-off",
)
axes[0, 0].legend()
axes[0, 1].legend()
for index in [1, 4, 7]:
    normalized_frequency = np.asarray(frequencies[1:] / carrier_corners[index])
    axes[1, 0].semilogx(
        normalized_frequency, 20 * np.log10(np.abs(pump_rf[index, 1:])), label=f"tau = {float(frequency_cases[index, 0]):g} ps"
    )
axes[1, 0].set(
    xlabel="Frequency / effective recovery corner",
    ylabel="Pump response / DC (dB)",
    title="Lifetime/loading change the corner frequency",
)
axes[1, 0].axvline(1, color="black", linestyle=":")
axes[1, 0].legend()
optical_transit_s = 3.6 * length_um * 1e-6 / 299792458
rf = np.asarray(optical_rf[case_index])
delayed = rf * np.exp(-2j * np.pi * np.asarray(frequencies) * optical_transit_s)
axes[1, 1].semilogx(ghz, np.unwrap(np.angle(rf[1:])) * 180 / np.pi, label="Moving-frame response")
axes[1, 1].semilogx(ghz, np.unwrap(np.angle(delayed[1:])) * 180 / np.pi, ":", label="With pure optical transit delay")
axes[1, 1].axvline(1 / (2 * np.pi * optical_transit_s) / 1e9, color="black", linestyle=":", alpha=0.5)
axes[1, 1].set(
    xlabel="Modulation frequency (GHz)",
    ylabel="RF phase (degrees)",
    title=f"Optical transit overlay: {optical_transit_s * 1e12:.1f} ps",
)
axes[1, 1].legend()
plt.show()
for index in [3, 4, 5]:
    print(
        f"CW {float(frequency_cases[index, 1]) * 1e3:g} mW: effective carrier corner {float(carrier_corners[index]) / 1e9:.3f} GHz"
    )
print("Nine operating points agree with the analytic recovery model; DC differential gain also agrees.")

### Time-domain check below, near, and above the recovery corner

A second fixture applies a 0.1% sinusoidal input-power modulation. Thirty-two cycles are simulated at each of three frequencies; only the last eight cycles are fitted. A sinusoid/cosinusoid fit extracts the complex input and output power phasors, including their relative phase. This uses actual nonlinear transients rather than treating the analytic curve as a simulation. The first 24 cycles allow carrier transients to settle, including at the highest frequency. The response is checked against the independent linearized circuit result.

This is a numerical cross-check of the optical small-signal transfer. It does not assume large-signal modulation or replace the MZM/VNA measurement with electrical pumping.


In [ ]:
@source(ports=("p1", "p2"), states=("i_src",), holomorphic=True)
def ModulatedCW(signals: Signals, t: float, power: float = 2e-4, frequency: float = 1e9, modulation: float = 1e-3) -> PhysicsReturn:
    field = jnp.sqrt(power * (1 + modulation * jnp.sin(2 * jnp.pi * frequency * t)))
    return {"p1": signals.i_src, "p2": -signals.i_src, "i_src": signals.p1 - signals.p2 - field}, {}


check_circuit = compile_circuit(
    netlist,
    {"soa": SemiconductorOpticalAmplifier, "bias": VoltageSource, "light": ModulatedCW, "load": Resistor, "ground": lambda: 0},
    is_complex=True,
    backend="dense",
    rtol=1e-10,
    atol=1e-12,
)
check_frequencies = carrier_corners[case_index] * jnp.array([0.05, 1.0, 20.0])
check_parameters = frequency_params(*frequency_cases[case_index])


def modulation_transient(frequency):
    params = {**check_parameters, "LIGHT.frequency": frequency}
    initial = check_circuit.dc(params={**params, "LIGHT.modulation": 0.0})
    step = 1 / (frequency * 64)
    times = jnp.arange(32 * 64 + 1) * step
    sol = check_circuit.transient(
        t0=0.0, t1=32 / frequency, dt0=step, y0=initial, saveat=times, params=params, max_steps=32 * 64 + 16
    )
    return times, jnp.abs(check_circuit.port(sol.ys, "input")) ** 2, jnp.abs(check_circuit.port(sol.ys, "output")) ** 2


check_times, check_input, check_output = jax.jit(jax.vmap(modulation_transient))(check_frequencies)
reference_rf = frequency_circuit.small_signal(params=check_parameters)(
    freqs=check_frequencies, sources={"LIGHT": 0.001 * jnp.sqrt(check_parameters["LIGHT.power"]) / 2}
)
reference_transfer = reference_rf.power("output", field_unit="sqrt(W)") / (
    cw_gains[case_index] * reference_rf.power("input", field_unit="sqrt(W)")
)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.7))
for k, ratio in enumerate([0.05, 1.0, 20.0]):
    time_window = np.asarray(check_times[k, -8 * 64 - 1 : -1])
    pin_window = np.asarray(check_input[k, -8 * 64 - 1 : -1])
    pout_window = np.asarray(check_output[k, -8 * 64 - 1 : -1])
    omega_t = 2 * np.pi * float(check_frequencies[k]) * time_window
    basis = np.column_stack([np.ones(len(time_window)), np.cos(omega_t), np.sin(omega_t)])
    input_fit = np.linalg.lstsq(basis, pin_window, rcond=None)[0]
    output_fit = np.linalg.lstsq(basis, pout_window, rcond=None)[0]
    # With exp(+j*omega*t), phasor = cosine coefficient - j*sine coefficient.
    measured = (output_fit[1] - 1j * output_fit[2]) / (input_fit[1] - 1j * input_fit[2]) / float(cw_gains[case_index])
    np.testing.assert_allclose(measured, reference_transfer[k], rtol=0.015, atol=0.003)
    last = slice(-2 * 64, None)
    axes[k].plot(np.arange(128) / 64, (pin_window[last] / input_fit[0] - 1) / 0.001, label="Input")
    axes[k].plot(np.arange(128) / 64, (pout_window[last] / output_fit[0] - 1) / 0.001, label="Output")
    axes[k].set(title=f"f/f_c = {ratio:g}", xlabel="Time (modulation periods)", ylabel="Fractional power modulation / 0.001")
    print(f"f/f_c={ratio:g}: normalized response {abs(measured):.4f}, phase {np.angle(measured, deg=True):.2f} deg")
axes[0].legend()
plt.show()
print("Nonlinear transient fits agree with small-signal response below, at, and above the recovery corner.")

## Numerical anchors and explicit noise bandwidth

Before using the batch results for design decisions, check the CW balance and the constant-gain limit. An independent scalar root verifies DC compression, and a short-pulse extraction law provides an energy benchmark. Exact 3.000 dB compression is used here, rather than the half-power approximation.

Output ASE is sampled with a fixed seed, outside the nonlinear residual. For one polarization, $S_{ASE}=n_{sp}\max(G-1,0)h_P\nu$ with the high-gain approximation $n_{sp}=10^{NF/10}/2$. Sample variance is $S_{ASE}/\Delta t$ over complex bandwidth $1/\Delta t$. A physical receiver filter is needed to obtain receiver-band noise. The noise check validates this stated bandwidth rather than interpreting timestep-wide noise as detected noise.


In [ ]:
ref_params = design_params(reference_design)
ref_current = reference_current_a * length_um / 600
h0 = float(unsaturated_gain(ref_current, length_um=length_um, lifetime_s=lifetime_s))
G0 = np.exp(h0)
input_powers = jnp.geomspace(2e-8, 6e-3, 32)
ops = circuit.dc(params={**ref_params, "LIGHT.pulse_energy": 0.0, "LIGHT.power": input_powers})
measured_h = np.real(circuit.port(ops, "SOA,h"))
roots = [brentq(lambda h, p=float(p): h - h0 + p / internal_saturation_power * np.expm1(h), 0.0, h0) for p in input_powers]
np.testing.assert_allclose(measured_h, roots, atol=2e-7)
linear = circuit.dc(
    params={**ref_params, "SOA.saturation": 0.0, "SOA.alpha_h": 0.0, "LIGHT.pulse_energy": 0.0, "LIGHT.power": 2e-6}
)
np.testing.assert_allclose(circuit.port(linear, "output"), np.sqrt(G0 * 2e-6), rtol=2e-7, atol=1e-10)
g3 = G0 * 10 ** (-3 / 10)
pout3 = internal_saturation_power * np.log(G0 / g3) / (g3 - 1) * g3
print(f"Reference weak-signal gain {10 * np.log10(G0):.2f} dB; 3 dB output compression {10 * np.log10(pout3 / 1e-3):.2f} dBm")

noise_dt = 0.75e-12
noise, expected = ase_field(
    np.full(100_000, G0), 1 / noise_dt, noise_figure_db=noise_figure_db, wavelength_nm=wavelength_nm, rng=np.random.default_rng(83)
)
np.testing.assert_allclose(np.mean(np.abs(noise) ** 2), expected.mean(), rtol=0.015)
fig, axes = plt.subplots(1, 2)
axes[0].semilogx(np.asarray(input_powers) * 1e3, measured_h * 10 / np.log(10))
axes[0].set(xlabel="CW input (mW)", ylabel="Gain (dB)")
noise_powers = np.abs(noise) ** 2
axes[1].hist(noise_powers / expected, bins=80, density=True, alpha=0.7, label="ASE samples")
x = np.linspace(0, 8, 300)
axes[1].plot(x, np.exp(-x), "--", label="Circular complex Gaussian power law")
axes[1].set(xlim=(0, 8), xlabel="Noise sample power / expected power", ylabel="Probability density")
axes[1].legend()
plt.show()
# Validate the short-pulse energy scale at the reference design.
reference_energy_out = float(evaluate(reference_design)[2])
u = float(reference_design[2]) * 1e-12 / saturation_energy
no_replenishment_pj = saturation_energy * np.log1p(G0 * np.expm1(u)) * 1e12
assert abs(reference_energy_out / no_replenishment_pj - 1) < 0.08
print("CW, linear limit, pulse-energy scale, batch independence, refinement, chirp, gradients, and ASE checks passed.")

The useful outputs of this notebook are the design maps, the synchronized performance measurements, and the sensitivities. Change the geometry or parameter grids and rerun to explore another design region. Change the device backend to assess CPU/GPU throughput, keeping compilation and warm execution separate. Larger grids can be processed in chunks if storing every waveform becomes memory-limited; returning only metrics reduces retained output, but does not remove all transient/adjoint memory costs.